# Lab 3.1: Coordinate Parallel Agent Work

**Day 1 - Session 3**

Goal: Complete two bounded workers, review durable handoffs, and machine-check a human-approved integration decision.

> Open this notebook in Google Colab:
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/agent-orchestration-companion/blob/main/day1/lab3_1_parallel-agent-coordination-colab/start/lab3_1_parallel-agent-coordination-colab.ipynb)

In [ ]:
# This lab uses only the Python standard library; no package installation is required.
import sys
print(f"Python {sys.version_info.major}.{sys.version_info.minor} ready; no API key required.")

In [ ]:
import hashlib
import json
from copy import deepcopy
from html import escape

## Section 1: Read the coordination contract

The email and SMS workers may proceed in parallel because they own different implementation and handoff files. Both consume the same locked shipment-event schema. Integration begins only after a human reviews their bounded changes and focused evidence.

| Worker | Owned files | Focused requirement |
| :--- | :--- | :--- |
| `email-renderer` | `channels/email.py`, `handoffs/email.json` | Escape untrusted HTML |
| `sms-renderer` | `channels/sms.py`, `handoffs/sms.json` | Normalize control whitespace and stay within 160 characters |

This Colab adaptation uses isolated functions and evidence records. The companion terminal lab uses real worktrees, commits, branches, and merges.

In [ ]:
BASE_SCHEMA = {
    "version": 1,
    "required": ["order_id", "customer_name", "carrier", "tracking_url"],
}
EVENT = {
    "order_id": "A-1042",
    "customer_name": "Riley",
    "carrier": "Parcel Express",
    "tracking_url": "https://track.example/A-1042",
}
EMAIL_COMMAND = "python3 -m unittest tests.test_email tests.test_security.ChannelSecurityTest.test_email_escapes_untrusted_html"
SMS_COMMAND = "python3 -m unittest tests.test_sms tests.test_security.ChannelSecurityTest.test_sms_normalizes_control_whitespace"
EXPECTED_FILES = {
    "email": {"channels/email.py", "handoffs/email.json"},
    "sms": {"channels/sms.py", "handoffs/sms.json"},
}
print("Contract loaded: two parallel workers, one locked schema, one human gate.")

## Section 2: Load the focused and orchestration checks

Focused checks turn each worker claim into observable evidence. The final validator checks ownership, shared-base alignment, approval timing, merge order, combined behavior, schema preservation, and the conflict decision.

In [ ]:
def contains_todo(value):
    if isinstance(value, str):
        return "todo" in value.lower()
    if isinstance(value, list):
        return any(contains_todo(item) for item in value)
    if isinstance(value, dict):
        return any(contains_todo(item) for item in value.values())
    return False


def schema_digest(schema):
    payload = json.dumps(schema, sort_keys=True).encode()
    return hashlib.sha256(payload).hexdigest()


def assert_email(renderer):
    rendered = renderer(EVENT)
    assert rendered["subject"] == "Order A-1042 shipped"
    assert all(value in rendered["body"] for value in ("Riley", "Parcel Express", "https://track.example/A-1042"))
    hostile = dict(EVENT, customer_name="<script>alert(1)</script>")
    hostile_body = renderer(hostile)["body"]
    assert "<script>" not in hostile_body and "&lt;script&gt;" in hostile_body


def assert_sms(renderer):
    rendered = renderer(EVENT)
    assert all(value in rendered for value in ("A-1042", "Parcel Express", "https://track.example/A-1042"))
    hostile = dict(EVENT, carrier="Parcel\n\tExpress", tracking_url="https://track.example/" + "x" * 180)
    hostile_message = renderer(hostile)
    assert not any(character in hostile_message for character in "\n\r\t")
    assert len(hostile_message) <= 160


def run_focused_check(name, assertion, renderer):
    try:
        assertion(renderer)
        return {"status": "passed", "summary": f"{name} behavior and security assertions passed."}
    except Exception as error:
        return {"status": "failed", "summary": f"{type(error).__name__}: {error}"}


def validate_handoff(name, handoff):
    errors = []
    expected_task = f"{name}-renderer"
    expected_command = EMAIL_COMMAND if name == "email" else SMS_COMMAND
    if contains_todo(handoff):
        errors.append(f"{name} handoff contains TODO evidence.")
    if handoff.get("task_id") != expected_task:
        errors.append(f"{name} handoff has the wrong task ID.")
    if handoff.get("base_commit") != "lab-base-v1":
        errors.append(f"{name} handoff does not share the approved base.")
    if len(handoff.get("implementation_commit", "")) < 7:
        errors.append(f"{name} handoff needs an implementation reference.")
    if set(handoff.get("changed_files", [])) != EXPECTED_FILES[name]:
        errors.append(f"{name} changed-file inventory violates ownership.")
    verification = handoff.get("verification", {})
    if verification.get("command") != expected_command or verification.get("status") != "passed":
        errors.append(f"{name} handoff lacks passing focused evidence.")
    if not handoff.get("decisions"):
        errors.append(f"{name} handoff must record an implementation decision.")
    return errors

## Section 3: Complete the isolated workers

Treat these functions as outputs from two parallel agents. Each worker must satisfy only its own task card and must not alter the shared schema or the other worker.

In [ ]:
def render_email(event):
    """Return a subject and HTML-safe body for one shipment event."""
    values = {key: escape(str(value), quote=True) for key, value in event.items()}
    return {
        "subject": f"Order {values['order_id']} shipped",
        "body": (
            f"Hello {values['customer_name']}, your order shipped via {values['carrier']}. "
            f"Track it at {values['tracking_url']}"
        ),
    }


def render_sms(event):
    """Return a normalized shipment message of no more than 160 characters."""
    values = {key: " ".join(str(value).split()) for key, value in event.items()}
    message = (
        f"Order {values['order_id']} shipped via {values['carrier']}. "
        f"Track: {values['tracking_url']}"
    )
    return message[:160]

## Section 4: Run focused checks

Run each worker's supplied check independently. A passing result is necessary evidence, but integration must still wait for handoff review and human approval.

In [ ]:
email_check = run_focused_check("email", assert_email, render_email)
sms_check = run_focused_check("sms", assert_sms, render_sms)
print("Email focused check:", email_check)
print("SMS focused check:", sms_check)

## Section 5: Create durable handoffs

Each handoff must identify the shared base, one implementation reference, the exact owned files, a concrete decision, the supplied command, and the observed focused result.

In [ ]:
email_handoff = {
    "task_id": "email-renderer",
    "base_commit": "lab-base-v1",
    "implementation_commit": "email7f31a2",
    "changed_files": ["channels/email.py", "handoffs/email.json"],
    "decisions": ["Used standard-library HTML escaping for every untrusted event value."],
    "verification": {"command": EMAIL_COMMAND, **email_check},
    "unresolved_risks": [],
    "next_checkpoint": "Human review before integration.",
}

sms_handoff = {
    "task_id": "sms-renderer",
    "base_commit": "lab-base-v1",
    "implementation_commit": "sms9c42e1",
    "changed_files": ["channels/sms.py", "handoffs/sms.json"],
    "decisions": ["Normalized control whitespace before enforcing the 160-character limit."],
    "verification": {"command": SMS_COMMAND, **sms_check},
    "unresolved_risks": [],
    "next_checkpoint": "Human review before integration.",
}

handoff_errors = validate_handoff("email", email_handoff) + validate_handoff("sms", sms_handoff)
print("Handoff review:", "ready" if not handoff_errors else handoff_errors)

## Section 6: Apply the human approval gate

The reviewer checks both file inventories, both focused results, the shared base, and the unchanged schema. Approval must be recorded before the second integration step.

In [ ]:
candidate_schema = deepcopy(BASE_SCHEMA)
schema_unchanged = schema_digest(candidate_schema) == schema_digest(BASE_SCHEMA)
review_errors = validate_handoff("email", email_handoff) + validate_handoff("sms", sms_handoff)
if not schema_unchanged:
    review_errors.append("The locked shipment-event schema changed.")

approval = {
    "approved": not review_errors and schema_unchanged,
    "evidence": "Reviewed both handoffs, bounded files, focused checks, shared base, and unchanged schema.",
    "timing": "before-second-integration",
}
print("Human gate:", approval)

## Section 7: Record integration and resolve the prepared conflict

Integrate in the approved email-then-SMS order. The prepared alternatives disagree about channel order; preserve the approved `('email', 'sms')` sequence rather than accepting either conflicting variant.

In [ ]:
conflict_variants = [("email", "sms", "email"), ("sms", "email")]
resolved_channel_order = ("email", "sms")

integration_record = {
    "workspace_assignments": {"agent/email": "colab-worker-email", "agent/sms": "colab-worker-sms"},
    "merge_order": ["agent/email", "agent/sms"],
    "approved_before_second_merge": approval["approved"],
    "approval_evidence": approval["evidence"],
    "full_check": {"command": "run both focused checks", "status": "passed"},
    "security_check": {"command": "rerun focused security assertions", "status": "passed"},
    "conflict_resolution": "Preserved the approved email-then-SMS channel order without accepting either conflict variant.",
}

## Section 8: Validate the orchestration record

This final check distrusts completion claims and reruns behavior. Repair every reported mismatch until the record is valid.

In [ ]:
def validate_orchestration():
    errors = validate_handoff("email", email_handoff) + validate_handoff("sms", sms_handoff)
    if email_handoff.get("base_commit") != sms_handoff.get("base_commit"):
        errors.append("Worker handoffs do not share a base.")
    if approval.get("approved") is not True or approval.get("timing") != "before-second-integration":
        errors.append("Human approval before the second integration is missing.")
    if contains_todo(approval.get("evidence", "")):
        errors.append("Human approval lacks reviewed evidence.")
    if integration_record.get("merge_order") != ["agent/email", "agent/sms"]:
        errors.append("Merge order must be agent/email then agent/sms.")
    if integration_record.get("approved_before_second_merge") is not True:
        errors.append("Integration record omits approval before the second merge.")
    if contains_todo(integration_record):
        errors.append("Integration record contains TODO evidence.")
    if integration_record.get("full_check", {}).get("status") != "passed":
        errors.append("Combined check is not recorded as passed.")
    if integration_record.get("security_check", {}).get("status") != "passed":
        errors.append("Security check is not recorded as passed.")
    if schema_digest(candidate_schema) != schema_digest(BASE_SCHEMA):
        errors.append("The locked shipment-event schema changed.")
    if resolved_channel_order != ("email", "sms") or resolved_channel_order in conflict_variants:
        errors.append("Conflict resolution must preserve email then SMS.")
    rerun_email = run_focused_check("email", assert_email, render_email)
    rerun_sms = run_focused_check("sms", assert_sms, render_sms)
    if rerun_email["status"] != "passed" or rerun_sms["status"] != "passed":
        errors.append("Combined behavior fails on rerun.")
    return errors


errors = validate_orchestration()
if errors:
    print("ORCHESTRATION RECORD INVALID")
    for error in errors:
        print(f"- {error}")
else:
    print("ORCHESTRATION RECORD VALID")
    print("Parallel tasks: email-renderer, sms-renderer")
    print("Human checkpoint: approved before integration")
    print("Merge order: agent/email -> agent/sms")
    print("Conflict decision: preserve email -> sms")
    print("Lab complete.")
    print("Takeaway: Parallel speed is trustworthy only when ownership, evidence, approval, and integration checks remain explicit.")